In [ ]:

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

In [1]:
from openai import OpenAI
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [2]:
message = "Hello, Llama! This is my first ever message to you! Hi!"
response = openai.chat.completions.create(model="llama3.2", messages=[{"role":"user", "content":message}])
print(response.choices[0].message.content)

It's great to meet you! I'm thrilled to be your first conversation partner, and I'm excited to chat with you. Don't worry if you're not sure where to start - we can take things one step at a time. How's your day going so far? Is there something specific on your mind that you'd like to talk about, or are you open to just seeing where the conversation takes us? I'm all ears (or rather, all text)!


In [14]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [15]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [12]:

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [16]:
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [18]:
summarize("http://mongodb.com/docs/")

'# Summary of the MongoDB Docs Website\n======================================\n\nThe MongoDB Docs website provides comprehensive documentation for MongoDB, a document-oriented, operational database. Key features and capabilities include:\n\n- Strong consistency with ACID transactions\n- Modern query capabilities such as geospatial search, lexical search, and vector search\n- Serverless horizontal scaling with geography-aware fault tolerance across all major clouds\n- Security primitives for enterprise environments\n\n**Announcements/New Features**\n\nThe website currently provides a **Start Here!** guide for deploying the first database and downloading necessary tools. It also lists several key features of the latest version, **MongoDB 8.0**, which includes improved AI capabilities.\n\n### News/Press Releasre\n- [Read Press Releases and News Stories](https://mongodb.com/newsroom): \nRecent articles include updates on how MongoDB is helping modernize various industries and applications